# Notebook 10 — DINOv2 Base, eight classes, 224px

This is one new model, with an unknown Kaggle score. It uses the public pretrained `vit_base_patch14_dinov2.lvd142m` transformer instead of another ConvNeXt or EfficientNet. The current confirmed best remains **0.9806**. No Kaggle submission is made by this notebook.

**Inputs:** upload `10_dinov2_inputs.zip` as a **private** Kaggle dataset and attach the original **Competition** images. Enable a GPU and Internet. No secret is required for these public weights. An existing optional Hugging Face secret can be enabled in the first cell. Do not install or upgrade packages automatically.

**Run one cell at a time.** After the short speed check in Part 3, share its output before starting the training cell. After Part 5, share the validation comparison before running Part 6. Part 6 exports test probabilities, one unscored candidate CSV and a backup; it does not upload that CSV to Kaggle.

The saved split 1337 has 54,098 training and 9,547 validation images. Eight epochs are fixed: one epoch trains only the new classifier, followed by seven epochs of low-rate full fine-tuning. Clean validation accuracy selects the checkpoint, with ordinary NLL as the tie breaker. Validation labels are never used for training updates. No full-data model is scored on this holdout.

We keep the existing square-resize pipeline, including mild geometry and downscaling augmentation. The model card's default size is 518px; **224px is an explicit experiment choice** to reduce cost. timm resamples the pretrained position embeddings when loading at this size. This is a combined architecture/pretraining/adaptation experiment, not a pure architecture ablation.

The primary validation candidate is fixed in advance: **75% split-trained reference proxy + 25% DINOv2 flip-averaged probabilities**. Test uses **75% exact 0.9806 winning probabilities + 25% DINOv2 flip-averaged probabilities**. The proxy is not the exact full-data winner. The reused split and synthetic stress tests cannot establish a higher hidden-label score. Do not spend a submission slot just because one validation prediction improves.

Completed epochs save `last.pth` (model, optimizer, scheduler, scaler and random states), `best.pth`, a CSV history, and a recovery ZIP. To keep them through a Kaggle reset, download the recovery ZIP or save a version with outputs. The final review ZIP omits `last.pth` to keep its download smaller; it preserves the selected model and all exported results. A new session cannot recover files that were never saved outside the old session.

Primary sources: [DINOv2 model](https://huggingface.co/timm/vit_base_patch14_dinov2.lvd142m), [DINOv2 paper](https://arxiv.org/abs/2304.07193), [linear probing then fine-tuning](https://arxiv.org/abs/2202.10054). These motivate the experiment; they do not predict its competition score.

In [ ]:
# Optional authentication. The token value is never printed or saved.
import os
USE_HF_SECRET = False
if USE_HF_SECRET:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('kaggle-assignment3')

from pathlib import Path
from io import BytesIO
import copy, hashlib, json, math, platform, random, shutil, time, zipfile
import numpy as np
import pandas as pd
from PIL import Image
import PIL
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
import timm
from timm.data import resolve_model_data_config
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from IPython.display import display

# Normally leave these paths unset; discovery checks for ambiguity.
BASE = None
INPUT_PACK = None   # An unpacked pack directory or 10_dinov2_inputs.zip.
RECOVERY_DIR = None # Optional directory containing this run's saved last.pth/best.pth.
MODEL_NAME = 'vit_base_patch14_dinov2.lvd142m'
RUN_NAME = 'dinov2_base224_lpft_val1337_seed42'
OUT = Path('/kaggle/working/dinov2_base224')
RUN_DIR = OUT/RUN_NAME
TRAIN_SEED = 42
SPLIT_SEED = 1337
EPOCHS = 8
BATCH_SIZE = 32
GRAD_ACCUM = 2
NUM_WORKERS = 0
IMAGE_SIZE = 224
MAX_TRAIN_HOURS = 8.0
CLASSES = ['CatEye','CatFace','CatMouth','CatNose','DogEye','DogFace','DogMouth','DogNose']
CLASS_TO_IDX = {c:i for i,c in enumerate(CLASSES)}
EXPECTED_LABEL_SHA = '94124e856786379a9c1e9c8c7ba4599a9f27fe8dc6ea4165dc4afc55024d41a3'
EXPECTED_SPLIT_HASH = '842af9ee7beb6fe8c85fd55163deeb15cf8a5c778506c075f987e9b9ed9889b6'
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before starting this notebook.'
assert MODEL_NAME in timm.list_models(pretrained=True), 'The recorded pretrained model is unavailable; review timm without upgrading it automatically.'
device=torch.device('cuda')
versions=dict(python=platform.python_version(),torch=str(torch.__version__),
              torchvision=str(torchvision.__version__),timm=str(timm.__version__),
              numpy=str(np.__version__),pandas=str(pd.__version__),pillow=str(PIL.__version__))
probe=timm.create_model(MODEL_NAME,pretrained=False,num_classes=len(CLASSES),img_size=IMAGE_SIZE)
resolved=resolve_model_data_config(probe)
assert np.allclose(resolved['mean'],[.485,.456,.406]) and np.allclose(resolved['std'],[.229,.224,.225]), 'Pretrained normalisation changed; review before training.'
assert tuple(probe.patch_embed.img_size)==(IMAGE_SIZE,IMAGE_SIZE)
assert probe.num_classes==8 and probe.head.out_features==8
del probe
print('GPU:',torch.cuda.get_device_name(0),'| versions:',versions,flush=True)
print('DINOv2 Base: 224px, eight classes, split 1337. One head epoch + seven full epochs.',flush=True)
print('Micro-batch 32, accumulation 2, effective batch 64. Workers: 0.',flush=True)


## 1 Verify the inputs

Reuse the completed data audit and exact split, and check a fixed sample of original image bytes. This does not repeat the whole image audit. The pack has saved predictions, not hidden test labels. It also verifies that the supplied winning probabilities reproduce the exact 0.9806 CSV.

In [ ]:
EXPECTED_PACK_MANIFEST_SHA = '3d6c24b37ed0c60c95ef1df4548e9b6d8dd513c4650454e6dc345fc2596269be'

def sha_bytes(raw): return hashlib.sha256(raw).hexdigest()

def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def read_pack(location):
    location=Path(location)
    if location.is_dir():
        def read(name): return (location/name).read_bytes()
    else:
        def read(name):
            with zipfile.ZipFile(location) as z: return z.read(name)
    manifest_raw=read('10_dinov2_inputs_manifest.json')
    assert sha_bytes(manifest_raw)==EXPECTED_PACK_MANIFEST_SHA, 'The input-pack manifest differs from the checked experiment pack.'
    manifest=json.loads(manifest_raw)
    content={}
    for name,record in manifest['files'].items():
        raw=read(name)
        assert len(raw)==record['bytes'] and sha_bytes(raw)==record['sha256'], f'Input-pack file failed verification: {name}'
        content[name]=raw
    return manifest,content,manifest_raw

def image_folder(base,kind):
    for path in [base/kind/kind,base/kind]:
        if (path/'00001.jpg').is_file(): return path
    raise FileNotFoundError(f'Missing {kind} images below {base}')

def probability_metrics(y,p):
    return dict(errors=int((p.argmax(1)!=y).sum()),accuracy=float((p.argmax(1)==y).mean()),
                nll=float(-np.log(p[np.arange(len(y)),y].clip(1e-12,1)).mean()))

def check_probabilities(p,rows):
    assert p.shape==(rows,len(CLASSES)) and np.isfinite(p).all() and (p>=0).all()
    assert np.allclose(p.sum(1),1,atol=2e-5,rtol=0)

def validate_frames(content,official_labels):
    meta=pd.read_csv(BytesIO(content['train_metadata_clean.csv']))
    tr=pd.read_csv(BytesIO(content['train_split_1337.csv']))
    va=pd.read_csv(BytesIO(content['val_split_1337.csv']))
    assert len(meta)==63645 and meta.image_id.is_unique and meta.md5.is_unique
    assert len(tr)==54098 and len(va)==9547 and tr.image_id.is_unique and va.image_id.is_unique
    assert set(tr.image_id).isdisjoint(set(va.image_id))
    assert set(tr.image_id)|set(va.image_id)==set(meta.image_id)
    official=official_labels.set_index('image_id')['class']
    assert np.array_equal(meta['class'],official.loc[meta.image_id])
    lookup=meta.set_index('image_id')
    for frame in [tr,va]:
        assert np.array_equal(frame['class'],lookup.loc[frame.image_id,'class'])
    assert set(lookup.loc[tr.image_id,'md5']).isdisjoint(set(lookup.loc[va.image_id,'md5']))
    assert sha_bytes(va.image_id.to_numpy(dtype='<i8').tobytes())==EXPECTED_SPLIT_HASH
    tr=tr.copy();va=va.copy()
    tr['label']=tr['class'].map(CLASS_TO_IDX).astype('int64')
    va['label']=va['class'].map(CLASS_TO_IDX).astype('int64')
    return meta,tr,va

if INPUT_PACK is None:
    pack_options=list(Path('/kaggle/input').rglob('10_dinov2_inputs_manifest.json'))
    pack_options=[p.parent for p in pack_options]
    if not pack_options: pack_options=list(Path('/kaggle/input').rglob('10_dinov2_inputs.zip'))
    assert len(pack_options)==1, f'Attach the private 10_dinov2_inputs pack or set INPUT_PACK explicitly. Found: {pack_options}'
    INPUT_PACK=pack_options[0]
pack,pack_files,pack_manifest_raw=read_pack(INPUT_PACK)
if BASE is None:
    data_options=[]
    for path in Path('/kaggle/input').rglob('train_labels.csv'):
        if not (path.parent/'sample_submission.csv').is_file(): continue
        try: image_folder(path.parent,'train');image_folder(path.parent,'test')
        except FileNotFoundError: continue
        data_options.append(path.parent)
    assert len(data_options)==1, f'Attach Competition or set BASE explicitly. Found: {data_options}'
    BASE=data_options[0]
BASE=Path(BASE)
TRAIN_DIR=image_folder(BASE,'train');TEST_DIR=image_folder(BASE,'test')
assert sha_file(BASE/'train_labels.csv')==EXPECTED_LABEL_SHA, 'The label file differs from the previously audited competition.'
labels=pd.read_csv(BASE/'train_labels.csv')
sample=pd.read_csv(BASE/'sample_submission.csv')
assert labels.columns.tolist()==sample.columns.tolist()==['image_id','class']
assert len(labels)==63660 and labels.image_id.is_unique and not labels.isna().any().any()
assert len(sample)==29443 and sample.image_id.is_unique and not sample.image_id.isna().any()
assert set(labels['class'])==set(CLASSES)
meta,train_df,val_df=validate_frames(pack_files,labels)
# Split CSVs contain IDs/classes only. Align image metadata without reordering either split.
metadata_lookup=meta.set_index('image_id')
for frame in [train_df,val_df]:
    for column in ['width','height','aspect']:
        frame[column]=metadata_lookup.loc[frame.image_id,column].to_numpy()
val_ids=val_df.image_id.to_numpy(dtype='int64');y=val_df.label.to_numpy(dtype='int64')
test_ids=sample.image_id.to_numpy(dtype='int64')
assert sha_bytes(test_ids.astype('<i8').tobytes())==pack['test_id_order_sha256'], 'Test IDs differ from the original sample order.'
refs={}
for name in ['control','geometry','v2']:
    ref_cfg=json.loads(pack_files[f'reference_{name}_config.json'])
    assert ref_cfg['split_hash']==EXPECTED_SPLIT_HASH and ref_cfg['split_seed']==1337 and not ref_cfg['full_data']
    for view in ['clean','tta']:
        with np.load(BytesIO(pack_files[f'reference_{name}_val_{view}.npz']),allow_pickle=False) as a:
            assert a['classes'].tolist()==CLASSES and str(a['split_hash'])==EXPECTED_SPLIT_HASH
            assert np.array_equal(a['image_id'],val_ids) and np.array_equal(a['labels'],y)
            p=a['probs'].copy()
        check_probabilities(p,len(y))
        assert probability_metrics(y,p)['errors']==pack['reference_errors'][f'{name}_{view}']
        refs[(name,view)]=p
# Verify original image bytes for the same fixed sample on each run.
for r in meta.sample(n=32,random_state=42).itertuples(index=False):
    raw=(TRAIN_DIR/f'{int(r.image_id):05d}.jpg').read_bytes()
    assert hashlib.md5(raw).hexdigest()==r.md5, 'An input image differs from the original audit.'
print('Verified:',len(train_df),'training /',len(val_df),'validation /',len(sample),'test images.',flush=True)
print('Reused the completed audit and exact split 1337; existing reference errors:',pack['reference_errors'],flush=True)


def read_reference(raw,ids,labels=None):
    with np.load(BytesIO(raw),allow_pickle=False) as a:
        assert a['classes'].tolist()==CLASSES and np.array_equal(a['image_id'],ids)
        if labels is not None: assert np.array_equal(a['labels'],labels)
        else: assert 'labels' not in a.files, 'Test reference must not include truth labels.'
        p=a['probs'].astype('float64')
    check_probabilities(p,len(ids));return p

reference_val=read_reference(pack_files['reference_validation_proxy.npz'],val_ids,y)
direct=.75*(.75*refs[('control','clean')]+.25*refs[('geometry','clean')])+.25*refs[('v2','tta')]
assert np.max(np.abs(direct-reference_val))<2e-7
assert np.array_equal(direct.argmax(1),reference_val.argmax(1))
assert probability_metrics(y,reference_val)['errors']==23
reference_test=read_reference(pack_files['reference_best_test.npz'],test_ids)
best_csv=pd.read_csv(BytesIO(pack_files['reference_best_submission.csv']))
assert sha_bytes(pack_files['reference_best_submission.csv'])==pack['best_submission_sha256']
assert np.array_equal(best_csv.image_id,test_ids)
assert np.array_equal(best_csv['class'],np.array(CLASSES)[reference_test.argmax(1)])
print('Exact 0.9806 test reference verified. Development proxy: 23 errors; not the full winner.',flush=True)


## 2 Save the settings and define image loading

The square resize, downscaling and mild geometry come from the existing training pipeline. Nothing is cropped away deliberately by the padding augmentation. DINOv2 uses ImageNet normalisation. The classifier starts with one head-only epoch at 0.001. Full fine-tuning then uses backbone rate 0.00002 and head rate 0.0002, weight decay 0.05, label smoothing 0.1, and cosine decay after a short warm-up. Accumulation averages the actual number of images in the final partial update correctly.

In [ ]:
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed); random.seed(worker_seed)

class RandomDownscale:
    def __init__(self, p=.3): self.p=p
    def __call__(self, img):
        if random.random() < self.p:
            target=random.randint(16,48); short=min(img.size)
            if short > target:
                scale=target/short
                img=img.resize((max(1,round(img.width*scale)), max(1,round(img.height*scale))),
                               Image.Resampling.BILINEAR)
        return img

class AxisPad:
    def __init__(self, p=.5, max_fraction=.15, fixed=None):
        self.p, self.max_fraction, self.fixed = p, max_fraction, fixed
    def __call__(self, img):
        if self.fixed is None:
            if random.random() >= self.p: return img
            fx=random.uniform(0,self.max_fraction)
            fy=random.uniform(0,self.max_fraction)
        else:
            fx,fy=self.fixed
        w,h=img.size
        px,py=round(w*fx),round(h*fy)
        # This runs after square resizing: padding is always smaller than either side.
        return TF.pad(img, [px,py,px,py], padding_mode='reflect')

def transforms_for(cfg, training=False, stress=None):
    s=cfg['size']; ops=[]
    if training: ops.append(RandomDownscale(cfg['downscale_p']))
    ops.append(T.Resize((s,s), interpolation=T.InterpolationMode.BILINEAR, antialias=True))
    if training and cfg['geometry']:
        ops.extend([AxisPad(), T.Resize((s,s), antialias=True)])
    if stress:
        fixed={'wide':(0,.15), 'tall':(.15,0)}[stress]
        ops.extend([AxisPad(fixed=fixed), T.Resize((s,s), antialias=True)])
    if training:
        ops.extend([T.RandomHorizontalFlip(.5),
                    T.RandomAffine(degrees=10, translate=(.05,.05), scale=(.9,1.1)),
                    T.ColorJitter(brightness=.3, contrast=.3, saturation=.2)])
    ops.extend([T.ToTensor(), T.Normalize(cfg['mean'],cfg['std'])])
    return T.Compose(ops)

class Images(Dataset):
    def __init__(self, frame, folder, transform):
        self.frame=frame.reset_index(drop=True); self.folder=folder; self.transform=transform
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]; image_id=int(r['image_id'])
        with Image.open(self.folder/f'{image_id:05d}.jpg') as im:
            x=self.transform(im.convert('RGB'))
        y=int(r['label']) if 'label' in self.frame.columns else -1
        return x,y,image_id

PIPELINE_SOURCE_SHA = 'f04bbb60499628ed7827606f874d9707ddbee854a8f0400e51de17da26150be1'


CFG=dict(model=MODEL_NAME,size=IMAGE_SIZE,geometry=True,epochs=EPOCHS,head_epochs=1,
         batch_size=BATCH_SIZE,gradient_accumulation=GRAD_ACCUM,effective_batch_size=BATCH_SIZE*GRAD_ACCUM,
         head_warmup_lr=.001,backbone_lr=.00002,head_finetune_lr=.0002,
         full_warmup_epoch_fraction=.1,weight_decay=.05,label_smoothing=.1,clip_grad_norm=1.,
         drop_path_rate=.05,downscale_p=.3,mean=list(map(float,resolved['mean'])),std=list(map(float,resolved['std'])),
         split_seed=SPLIT_SEED,train_seed=TRAIN_SEED,full_data=False,classes=CLASSES,
         label_sha256=EXPECTED_LABEL_SHA,split_hash=EXPECTED_SPLIT_HASH,
         input_pack_manifest_sha256=EXPECTED_PACK_MANIFEST_SHA,
         experiment_version='10-dinov2-base224-lpft-v1',pipeline_source_sha256=PIPELINE_SOURCE_SHA,
         augmentation=dict(downscale_p=.3,downscale_short_side=[16,48],geometry_p=.5,
                           geometry_max_fraction_per_side=.15,horizontal_flip_p=.5,
                           affine_degrees=10,translation_per_axis=.05,scale=[.9,1.1],
                           brightness=.3,contrast=.3,saturation=.2),
         resize_interpolation='bilinear',antialias=True,num_workers=NUM_WORKERS,
         persistent_workers=False,drop_last=False,amp='float16',primary_new_model_weight=.25,
         selection='clean validation accuracy; ordinary NLL tie breaker',versions=versions)
CFG_SHA=sha_bytes(json.dumps(CFG,sort_keys=True).encode())

def make_model(pretrained=False):
    result=timm.create_model(MODEL_NAME,pretrained=pretrained,num_classes=len(CLASSES),
                             img_size=IMAGE_SIZE,drop_path_rate=CFG['drop_path_rate'])
    assert tuple(result.patch_embed.img_size)==(IMAGE_SIZE,IMAGE_SIZE)
    assert result.num_classes==8 and result.head.out_features==8
    return result.to(device)

def atomic_json(path,data):
    path=Path(path);temp=path.with_name(path.name+'.tmp')
    temp.write_text(json.dumps(data,indent=2)+'\n');os.replace(temp,path)

def atomic_torch(path,data):
    path=Path(path);temp=path.with_name(path.name+'.tmp')
    torch.save(data,temp);os.replace(temp,path)

def make_loader(frame,folder,training=False,stress=None):
    generator=torch.Generator().manual_seed(TRAIN_SEED)
    loader=DataLoader(Images(frame,folder,transforms_for(CFG,training=training,stress=stress)),
                      batch_size=BATCH_SIZE,shuffle=training,drop_last=False,
                      num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=False,
                      worker_init_fn=seed_worker,generator=generator)
    return loader,generator

def current_random_state(generator):
    n=np.random.get_state()
    return dict(python=random.getstate(),numpy=(n[0],n[1].tolist(),n[2],n[3],n[4]),
                torch=torch.get_rng_state(),cuda=torch.cuda.get_rng_state(device),
                loader=generator.get_state())

def restore_random_state(state,generator):
    random.setstate(state['python'])
    n=state['numpy'];np.random.set_state((n[0],np.array(n[1],dtype=np.uint32),n[2],n[3],n[4]))
    torch.set_rng_state(state['torch']);torch.cuda.set_rng_state(state['cuda'],device)
    generator.set_state(state['loader'])

def schedule_factor(step,warmup_steps,total_steps):
    if step<warmup_steps: return (step+1)/warmup_steps
    progress=(step-warmup_steps)/max(1,total_steps-warmup_steps)
    return .5*(1+math.cos(math.pi*min(progress,1)))

@torch.inference_mode()
def predict_views(model,loader,description,flip=False):
    model.eval();clean=[];tta=[];ys=[];ids=[]
    for x,target,image_id in tqdm(loader,desc=description,leave=True):
        x=x.to(device,non_blocking=True)
        with torch.autocast('cuda',dtype=torch.float16):
            logits=model(x)
        p=torch.softmax(logits.float(),dim=1)
        clean.append(p.cpu().numpy())
        if flip:
            with torch.autocast('cuda',dtype=torch.float16):
                flipped_logits=model(x.flip(-1))
            q=torch.softmax(flipped_logits.float(),dim=1)
            tta.append(((p+q)/2).cpu().numpy())
        ys.append(target.numpy());ids.append(image_id.numpy())
    return np.concatenate(clean),None if not flip else np.concatenate(tta),np.concatenate(ys),np.concatenate(ids)

def restore_saved_run():
    if (RUN_DIR/'config.json').exists():
        assert json.loads((RUN_DIR/'config.json').read_text())==dict(config_sha256=CFG_SHA,config=CFG), 'Existing run settings differ; they will not be overwritten.'
        if (RUN_DIR/'last.pth').exists() or (RUN_DIR/'training_complete.json').exists(): return
    existing=[p for p in RUN_DIR.glob('*') if p.suffix!='.tmp' and p.name!='config.json'] if RUN_DIR.exists() else []
    assert not existing, 'This local run directory lacks its config. Preserve it and review before restarting.'
    if RECOVERY_DIR is not None:
        choices=[Path(RECOVERY_DIR)]
    else:
        choices=[p.parent for p in Path('/kaggle/input').rglob('config.json') if p.parent.name==RUN_NAME]
    assert len(choices)<=1, f'Multiple saved Notebook 10 runs found. Set RECOVERY_DIR explicitly: {choices}'
    if choices:
        source=choices[0]
        record=json.loads((source/'config.json').read_text())
        assert record['config_sha256']==CFG_SHA, 'Saved run settings/environment differ; preserve that run and review the config.'
        assert (source/'last.pth').is_file() or (source/'training_complete.json').is_file(), 'Saved run has no recoverable epoch checkpoint.'
        shutil.copytree(source,RUN_DIR,dirs_exist_ok=True)
        print('Restored saved Notebook 10 files from:',source,flush=True)

restore_saved_run()
RUN_DIR.mkdir(parents=True,exist_ok=True)
record=dict(config_sha256=CFG_SHA,config=CFG)
if (RUN_DIR/'config.json').exists():
    assert json.loads((RUN_DIR/'config.json').read_text())==record, 'Existing run has different settings. It will not be overwritten.'
else: atomic_json(RUN_DIR/'config.json',record)
OUT.mkdir(parents=True,exist_ok=True)
for name in ['train_split_1337.csv','val_split_1337.csv']:
    target=OUT/name
    if target.exists(): assert target.read_bytes()==pack_files[name]
    else: target.write_bytes(pack_files[name])
atomic_json(OUT/'input_pack_manifest.json',pack)
print('Pretrained normalisation:',CFG['mean'],CFG['std'],'| output:',OUT,flush=True)


def export_backup(kind='review'):
    assert kind in ['review','recovery']
    target=OUT.parent/f'10_dinov2_{kind}_backup.zip'
    temp=target.with_name(target.name+'.tmp')
    with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
        for path in sorted(OUT.rglob('*')):
            if not path.is_file() or path.name.endswith('.tmp'): continue
            if kind=='review' and path.name=='last.pth': continue
            archive.write(path,path.relative_to(OUT.parent))
    with zipfile.ZipFile(temp) as archive:
        assert archive.testzip() is None, 'Backup CRC check failed; original output files are retained.'
    os.replace(temp,target)
    print(f'{kind.capitalize()} backup verified: {target} ({target.stat().st_size/1024**2:.1f} MiB)',flush=True)
    return target

print('Settings and helper functions ready. Next: run the short speed check, then share its output.',flush=True)


## 3 Short speed and memory check — stop here and share the output

This downloads the pretrained weights once and times full forward/backward passes on a synthetic batch of 32 images. **It makes no optimizer update and changes no model weights.** The estimate excludes image loading, validation, optimizer overhead, backups and test inference; it is not a promise of total run time. The first real full-training epoch gives a better estimate. Peak memory below excludes AdamW's later optimizer states. If this cell reports an error, keep the output and stop before training.

In [ ]:
def benchmark_model():
    seed_all(TRAIN_SEED)
    probe=make_model(pretrained=True)
    probe.train();probe.zero_grad(set_to_none=True)
    x=torch.randn(BATCH_SIZE,3,IMAGE_SIZE,IMAGE_SIZE,device=device)
    targets=torch.arange(BATCH_SIZE,device=device)%len(CLASSES)
    criterion=nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])
    # This transformer has no running BatchNorm statistics; no optimizer is constructed.
    for _ in range(2):
        with torch.autocast('cuda',dtype=torch.float16): loss=criterion(probe(x),targets)
        assert torch.isfinite(loss).item()
        loss.backward();probe.zero_grad(set_to_none=True)
    torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()
    started=time.perf_counter()
    for _ in range(6):
        with torch.autocast('cuda',dtype=torch.float16): loss=criterion(probe(x),targets)
        assert torch.isfinite(loss).item()
        loss.backward();probe.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    seconds=time.perf_counter()-started
    rate=6*BATCH_SIZE/seconds
    report=dict(config_sha256=CFG_SHA,gpu=torch.cuda.get_device_name(),synthetic_images_per_second=rate,
                forward_backward_compute_minutes_per_full_epoch=len(train_df)/rate/60,
                compute_hours_for_seven_full_epochs=7*len(train_df)/rate/3600,
                peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3,
                benchmark_excludes_io_validation_optimizer_and_backup=True,optimizer_updates=0)
    atomic_json(OUT/'speed_check.json',report)
    del probe,x,targets,loss,criterion
    torch.cuda.empty_cache();seed_all(TRAIN_SEED)
    print(json.dumps(report,indent=2),flush=True)
    print('Speed check complete. Stop here and share this output before running Part 4.',flush=True)
    return report

speed_check=benchmark_model()


## 4 Train or resume — only after reviewing the speed check

This cell trains the one model. Head-only training is epoch 1; epochs 2–8 fine-tune all layers. A phase change creates a new optimizer and scheduler intentionally; a resumed epoch in the same phase restores both. Schedulers advance only after successful AMP optimizer updates. An interrupted epoch repeats from the last saved epoch. A budget guard checks between epochs at eight cumulative training hours; it preserves files and stops for review if reached. It cannot measure Kaggle's account-wide GPU quota or time lost in an interrupted epoch.

Every completed epoch creates a recovery ZIP. Keep it outside the live session before ending Kaggle. It is larger than the final review ZIP because it includes the optimizer and last checkpoint. Clean validation NLL is ordinary cross-entropy; training loss uses label smoothing and augmentation, so the two losses have different definitions.

In [ ]:
def phase_for_epoch(epoch):
    return 'head' if epoch<=CFG['head_epochs'] else 'full'

def optimizer_for_phase(model,phase,updates_per_epoch):
    for name,p in model.named_parameters(): p.requires_grad_(phase=='full' or name.startswith('head.'))
    if phase=='head':
        opt=torch.optim.AdamW(model.head.parameters(),lr=CFG['head_warmup_lr'],weight_decay=CFG['weight_decay'])
        scheduler=torch.optim.lr_scheduler.LambdaLR(opt,lambda step:1.)
    else:
        backbone=[p for name,p in model.named_parameters() if not name.startswith('head.')]
        opt=torch.optim.AdamW([dict(params=backbone,lr=CFG['backbone_lr']),
                               dict(params=model.head.parameters(),lr=CFG['head_finetune_lr'])],weight_decay=CFG['weight_decay'])
        total=(EPOCHS-CFG['head_epochs'])*updates_per_epoch
        warmup=max(1,round(updates_per_epoch*CFG['full_warmup_epoch_fraction']))
        scheduler=torch.optim.lr_scheduler.LambdaLR(opt,lambda step:schedule_factor(step,warmup,total))
    return opt,scheduler

def train_or_resume():
    seed_all(TRAIN_SEED)
    completed=RUN_DIR/'training_complete.json';last_path=RUN_DIR/'last.pth';best_path=RUN_DIR/'best.pth'
    if completed.exists():
        done=json.loads(completed.read_text())
        assert done['config_sha256']==CFG_SHA and done['completed_epochs']==EPOCHS
        assert sha_file(best_path)==done['best_sha256']
        assert sha_file(RUN_DIR/'history.csv')==done['history_sha256']
        saved=torch.load(best_path,map_location='cpu',weights_only=True)
        assert saved['config_sha256']==CFG_SHA and saved['config']==CFG
        model=make_model(False);model.load_state_dict(saved['model'])
        history=pd.read_csv(RUN_DIR/'history.csv')
        assert history.epoch.tolist()==list(range(1,EPOCHS+1))
        print('Reused completed training. Selected epoch:',saved['epoch'],flush=True)
        return model,history
    assert last_path.exists() or not (best_path.exists() or (RUN_DIR/'history.csv').exists()), 'Partial run without last.pth: preserve files and review before restarting.'
    resume=last_path.exists()
    model=make_model(pretrained=not resume)
    train_loader,train_generator=make_loader(train_df,TRAIN_DIR,training=True)
    val_loader,_=make_loader(val_df,TRAIN_DIR)
    updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM)
    scaler=torch.amp.GradScaler('cuda')
    criterion=nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])
    history=[];best_rank=(-1.,-float('inf'));best_epoch=None;best_state=None;best_metrics=None
    skipped=0;updates=0;start_epoch=1;opt=None;scheduler=None;active_phase=None;resume_state=None
    prior_seconds=0.;session_start=time.perf_counter()
    if resume:
        resume_state=torch.load(last_path,map_location='cpu',weights_only=True)
        assert resume_state['config_sha256']==CFG_SHA and resume_state['config']==CFG
        assert 1<=resume_state['epoch']<=EPOCHS
        model.load_state_dict(resume_state['model']);scaler.load_state_dict(resume_state['scaler'])
        history=resume_state['history'];best_rank=tuple(resume_state['best_rank'])
        best_epoch=resume_state['best_epoch'];best_state=resume_state['best_model'];best_metrics=resume_state['best_metrics']
        assert [r['epoch'] for r in history]==list(range(1,resume_state['epoch']+1))
        skipped=resume_state['skipped_updates'];updates=resume_state['successful_updates'];start_epoch=resume_state['epoch']+1
        prior_seconds=resume_state['training_seconds']
        restore_random_state(resume_state['random_state'],train_generator)
        print('Resuming after saved epoch',start_epoch-1,'of',EPOCHS,flush=True)
    else:
        print('START: one DINOv2 Base; one head epoch + seven full epochs.',flush=True)
    for epoch in range(start_epoch,EPOCHS+1):
        total_seconds=prior_seconds+time.perf_counter()-session_start
        if total_seconds>=MAX_TRAIN_HOURS*3600:
            export_backup('recovery')
            raise RuntimeError('Eight-hour training guard reached between saved epochs. Download the recovery ZIP and share the status before continuing.')
        phase=phase_for_epoch(epoch)
        if phase!=active_phase:
            if opt is not None: del opt,scheduler
            opt,scheduler=optimizer_for_phase(model,phase,updates_per_epoch)
            if resume_state is not None and resume_state['phase']==phase:
                opt.load_state_dict(resume_state['optimizer']);scheduler.load_state_dict(resume_state['scheduler'])
            active_phase=phase
        resume_state=None
        started=time.perf_counter()
        model.train()
        if phase=='head': model.eval();model.head.train()
        opt.zero_grad(set_to_none=True)
        loss_sum=0.;correct=0;n=0
        bar=tqdm(train_loader,desc=f'DINOv2 {phase} epoch {epoch}/{EPOCHS}',leave=True)
        for batch_index,(x,target,_) in enumerate(bar):
            x=x.to(device,non_blocking=True);target=target.to(device,non_blocking=True)
            with torch.autocast('cuda',dtype=torch.float16): logits=model(x);loss=criterion(logits,target)
            assert torch.isfinite(loss).item(), 'Non-finite loss. Preserve the last completed epoch.'
            group_start=(batch_index//GRAD_ACCUM)*GRAD_ACCUM
            group_images=min(BATCH_SIZE*GRAD_ACCUM,len(train_df)-group_start*BATCH_SIZE)
            scaler.scale(loss*(len(target)/group_images)).backward()
            if (batch_index+1)%GRAD_ACCUM==0 or batch_index+1==len(train_loader):
                before=scaler.get_scale()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],CFG['clip_grad_norm'])
                scaler.step(opt);scaler.update()
                if scaler.get_scale()>=before: scheduler.step();updates+=1
                else: skipped+=1
                opt.zero_grad(set_to_none=True)
            loss_sum+=float(loss.item())*len(target)
            correct+=int((logits.argmax(1)==target).sum().item());n+=len(target)
            bar.set_postfix(loss=f'{loss_sum/n:.4f}',lr=f'{opt.param_groups[0]["lr"]:.2g}')
        assert n==len(train_df)
        p,_,target,ids=predict_views(model,val_loader,f'Clean validation {epoch}/{EPOCHS}')
        assert np.array_equal(ids,val_ids) and np.array_equal(target,y)
        check_probabilities(p,len(y));metrics=probability_metrics(y,p)
        rank=(metrics['accuracy'],-metrics['nll'])
        current_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        if rank>best_rank:
            best_rank=rank;best_epoch=epoch;best_state=current_state;best_metrics=metrics
        row=dict(epoch=epoch,phase=phase,train_loss=loss_sum/n,train_acc=correct/n,val_acc=metrics['accuracy'],
                 val_nll=metrics['nll'],val_errors=metrics['errors'],skipped_updates=skipped,
                 successful_updates=updates,seconds=round(time.perf_counter()-started,1))
        history.append(row)
        payload=dict(config_sha256=CFG_SHA,config=CFG,epoch=epoch,phase=phase,model=current_state,
                     optimizer=opt.state_dict(),scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),
                     history=history,best_model=best_state,best_rank=best_rank,best_epoch=best_epoch,best_metrics=best_metrics,
                     skipped_updates=skipped,successful_updates=updates,random_state=current_random_state(train_generator),
                     training_seconds=prior_seconds+time.perf_counter()-session_start)
        atomic_torch(last_path,payload);del payload,current_state
        pd.DataFrame(history).to_csv(RUN_DIR/'history.csv',index=False)
        atomic_torch(best_path,dict(model=best_state,config=CFG,config_sha256=CFG_SHA,epoch=best_epoch,validation=best_metrics))
        atomic_json(RUN_DIR/'epoch_status.json',dict(config_sha256=CFG_SHA,completed_epoch=epoch,best_epoch=best_epoch,
                    training_hours=(prior_seconds+time.perf_counter()-session_start)/3600))
        print(row,flush=True)
        export_backup('recovery')
        if epoch==2:
            print('First full epoch minutes:',round(row['seconds']/60,1),'| remaining full epochs:',EPOCHS-epoch,flush=True)
    assert len(history)==EPOCHS and best_state is not None
    pd.DataFrame(history).to_csv(RUN_DIR/'history.csv',index=False)
    atomic_torch(best_path,dict(model=best_state,config=CFG,config_sha256=CFG_SHA,epoch=best_epoch,validation=best_metrics))
    atomic_json(completed,dict(config_sha256=CFG_SHA,completed_epochs=EPOCHS,best_epoch=best_epoch,
                               best_sha256=sha_file(best_path),history_sha256=sha_file(RUN_DIR/'history.csv')))
    model.load_state_dict(best_state)
    del train_loader,val_loader,opt,scheduler,scaler,best_state
    torch.cuda.empty_cache();export_backup('recovery');export_backup('review')
    print('Training complete. Loaded selected checkpoint epoch',best_epoch,flush=True)
    return model,pd.DataFrame(history)

model,history=train_or_resume()
fig,axes=plt.subplots(1,2,figsize=(10,3))
axes[0].plot(history.epoch,history.train_acc,label='augmented training')
axes[0].plot(history.epoch,history.val_acc,label='clean validation')
axes[0].set_title('Accuracy');axes[0].set_xlabel('Epoch');axes[0].legend()
axes[1].plot(history.epoch,history.val_nll);axes[1].set_title('Ordinary validation NLL');axes[1].set_xlabel('Epoch')
plt.tight_layout();fig.savefig(RUN_DIR/'learning_curves.png',dpi=160);plt.show();plt.close(fig)


## 5 Compare errors — stop here and share the output

Compare the new model and the fixed 25% blend on the original development images. Count both corrected predictions and new mistakes. Report the class pairs and image-size groups, including the previously difficult small crops. Wide/tall ordinary passes are secondary synthetic checks, not hidden-test predictions. Their padding fraction is the same as earlier checks; the models use their own training resolutions.

Only the fixed 25% blend is the primary candidate. No weight grid or routing-threshold search is performed. The reference proxy has 23 original errors. A small gain on this reused split is weak evidence; review before generating test predictions.

In [ ]:
BEST_SHA=sha_file(RUN_DIR/'best.pth')

def checked_prediction(path,ids,labels=None):
    with np.load(path,allow_pickle=False) as a:
        assert a['classes'].tolist()==CLASSES and np.array_equal(a['image_id'],ids)
        assert str(a['config_sha256'])==CFG_SHA and str(a['checkpoint_sha256'])==BEST_SHA
        assert str(a['split_hash'])==EXPECTED_SPLIT_HASH
        assert str(a['prediction_kind'])==Path(path).stem
        if labels is not None: assert np.array_equal(a['labels'],labels)
        else: assert 'labels' not in a.files
        p=a['probs'].copy()
    check_probabilities(p,len(ids))
    return p

def save_prediction(path,p,ids,labels=None):
    check_probabilities(p,len(ids));temp=path.with_name(path.name+'.tmp')
    data=dict(probs=p.astype('float32'),image_id=np.asarray(ids,dtype='int64'),classes=np.array(CLASSES),
              config_sha256=np.array(CFG_SHA),checkpoint_sha256=np.array(BEST_SHA),
              split_hash=np.array(EXPECTED_SPLIT_HASH),prediction_kind=np.array(Path(path).stem))
    if labels is not None: data['labels']=np.asarray(labels,dtype='int64')
    with temp.open('wb') as f: np.savez_compressed(f,**data)
    os.replace(temp,path)
    return checked_prediction(path,ids,labels)

def predictions_for(frame,folder,prefix,labels=None):
    ids=frame.image_id.to_numpy(dtype='int64')
    paths={view:RUN_DIR/f'{prefix}_{view}.npz' for view in ['clean','tta']}
    found={view:checked_prediction(path,ids,labels) for view,path in paths.items() if path.exists()}
    if len(found)==2:
        print('Reused verified predictions:',prefix,flush=True);return found
    loader,_=make_loader(frame,folder)
    clean,tta,target,pred_ids=predict_views(model,loader,f'{prefix}: original + horizontal flip',flip=True)
    assert np.array_equal(pred_ids,ids)
    if labels is not None: assert np.array_equal(target,labels)
    for view,p in [('clean',clean),('tta',tta)]:
        if view not in found: found[view]=save_prediction(paths[view],p,ids,labels)
    del loader
    return found

new_val=predictions_for(val_df,TRAIN_DIR,'val',labels=y)
rows=[];changed=[]
old_pred=reference_val.argmax(1)
primary_weight=CFG['primary_new_model_weight']
for view in ['clean','tta']:
    candidate=(1-primary_weight)*reference_val+primary_weight*new_val[view].astype('float64')
    for name,p in [('reference_proxy',reference_val),('dinov2_alone',new_val[view]),('fixed_25pct_blend',candidate)]:
        pred=p.argmax(1)
        for group,mask in [('all_validation',np.ones(len(y),dtype=bool)),
                           ('cat_mouth_nose',np.isin(y,[2,3])),
                           ('short_side_under32',np.minimum(val_df['width'],val_df['height']).to_numpy()<32),
                           ('non_square',np.abs(val_df['aspect'].to_numpy()-1)>=.05)]:
            rows.append(dict(view=view,candidate=name,group=group,images=int(mask.sum()),
                        **probability_metrics(y[mask],p[mask]),
                        fixed=int(((old_pred!=y)&(pred==y)&mask).sum()),
                        broken=int(((old_pred==y)&(pred!=y)&mask).sum()),
                        changed=int(((old_pred!=pred)&mask).sum())))
    for i in np.flatnonzero(old_pred!=candidate.argmax(1)):
        changed.append(dict(view=view,image_id=int(val_ids[i]),true_class=CLASSES[y[i]],
                       reference_class=CLASSES[old_pred[i]],candidate_class=CLASSES[candidate.argmax(1)[i]]))
    for name,p in [('dinov2',new_val[view]),('blend25',candidate)]:
        (RUN_DIR/f'classification_report_{name}_{view}.txt').write_text(classification_report(y,p.argmax(1),labels=range(8),target_names=CLASSES,digits=5,zero_division=0))
        pd.DataFrame(confusion_matrix(y,p.argmax(1),labels=range(8)),index=CLASSES,columns=CLASSES).to_csv(RUN_DIR/f'confusion_matrix_{name}_{view}.csv')
table=pd.DataFrame(rows);table.to_csv(OUT/'original_validation_comparison.csv',index=False)
pd.DataFrame(changed,columns=['view','image_id','true_class','reference_class','candidate_class']).to_csv(OUT/'changed_validation_predictions.csv',index=False)
display(table[table.group=='all_validation']);display(table[(table.view=='tta')&(table.candidate=='fixed_25pct_blend')])
stress_rows=[]
for case in ['wide','tall']:
    path=RUN_DIR/f'val_{case}_clean.npz'
    if path.exists(): p=checked_prediction(path,val_ids,y)
    else:
        loader,_=make_loader(val_df,TRAIN_DIR,stress=case)
        p,_,target,ids=predict_views(model,loader,f'{case}: secondary ordinary check')
        assert np.array_equal(ids,val_ids) and np.array_equal(target,y)
        p=save_prediction(path,p,val_ids,y);del loader
    ref=read_reference(pack_files[f'reference_v2_val_{case}.npz'],val_ids,y)
    for name,probs in [('v2_reference',ref),('dinov2',p)]:
        stress_rows.append(dict(case=case,model=name,**probability_metrics(y,probs)))
stress_table=pd.DataFrame(stress_rows);stress_table.to_csv(OUT/'secondary_stress_comparison.csv',index=False);display(stress_table)
primary=table[(table.view=='tta')&(table.candidate=='fixed_25pct_blend')&(table.group=='all_validation')].iloc[0].to_dict()
atomic_json(OUT/'validation_review.json',dict(config_sha256=CFG_SHA,checkpoint_sha256=BEST_SHA,primary=primary,
            primary_weight=primary_weight,exact_winning_ensemble_independent_validation=None,public_score=None))
export_backup('review')
print('Validation complete. Share these tables and the review ZIP before Part 6. No test inference or CSV has been generated.',flush=True)


## 6 Export one unscored test candidate and the review backup

Run this cell only after reviewing Part 5. It keeps the exact 0.9806 winning probabilities at 75% and adds the new model's original/flip average at 25%, as specified before training. The sample CSV supplies image IDs only. It creates a candidate for manual review; it does not submit it. Keep the current best and the second remaining submission slot while we review the result.

In [ ]:
test_predictions=predictions_for(sample[['image_id']],TEST_DIR,'test')
weight=CFG['primary_new_model_weight']
candidate=(1-weight)*reference_test+weight*test_predictions['tta'].astype('float64')
check_probabilities(candidate,len(test_ids))
submission=pd.DataFrame({'image_id':test_ids,'class':np.array(CLASSES)[candidate.argmax(1)]})
assert submission.columns.tolist()==['image_id','class'] and len(submission)==29443
assert submission.image_id.is_unique and np.array_equal(submission.image_id,sample.image_id)
assert set(submission['class'])<=set(CLASSES) and not submission.isna().any().any()
name='submission_best75_dinov2b224tta25.csv'
path=OUT/name
raw=submission.to_csv(index=False,lineterminator='\n').encode()
if path.exists(): assert path.read_bytes()==raw, 'Existing candidate differs; it will not be overwritten.'
else: path.write_bytes(raw)
changed=int((submission['class']!=best_csv['class']).sum())
recipe=dict(candidate=name,submitted=False,public_score=None,source_best_public_score=.9806,
            weights=dict(best_09806=.75,dinov2_base224_tta=.25),new_model_config_sha256=CFG_SHA,
            new_model_checkpoint_sha256=BEST_SHA,source_best_submission_sha256=pack['best_submission_sha256'],
            source_best_probability_sha256=pack['files']['reference_best_test.npz']['sha256'],
            candidate_sha256=sha_file(path),changed_test_labels_from_best=changed,
            independent_exact_blend_validation=None)
recipe_path=OUT/'candidate_recipe.json'
if recipe_path.exists():
    previous=json.loads(recipe_path.read_text())
    assert previous['candidate_sha256']==recipe['candidate_sha256']
    recipe['submitted']=previous.get('submitted',False);recipe['public_score']=previous.get('public_score')
atomic_json(recipe_path,recipe)
prediction_files={p.name:sha_file(p) for p in RUN_DIR.glob('*.npz')}
atomic_json(RUN_DIR/'prediction_manifest.json',dict(config_sha256=CFG_SHA,checkpoint_sha256=BEST_SHA,
            files=prediction_files,classes=CLASSES,validation_images=len(val_df),test_images=len(sample)))
atomic_json(OUT/'complete_run.json',dict(run=RUN_NAME,config_sha256=CFG_SHA,best_sha256=BEST_SHA,
            current_best_public_score=.9806,new_model_public_score=None,epochs=EPOCHS,
            selected_epoch=json.loads((RUN_DIR/'training_complete.json').read_text())['best_epoch'],
            independent_winning_ensemble_validation=None))
archive=export_backup('review')
print('Candidate:',path,'| changed test labels:',changed,flush=True)
print('COMPLETE. Download:',archive,flush=True)
print('Share the backup for review. No Kaggle submission was made.',flush=True)
if changed==0: print('This candidate duplicates the current best labels. Do not spend a submission slot on it.',flush=True)
